# Ensemble Learning & Multi-Modal Fusion

## Ensemble Strategy
Train three architecturally diverse models and combine via **soft-voting**:

```
EfficientNet-B4 ─┐
ResNet-50+CBAM  ─┼─► Average Probabilities ─► Final Prediction
DenseNet-169    ─┘
```

Diversity is key: different inductive biases → uncorrelated errors → lower ensemble error.

## Multi-Modal Fusion
Extend the best single model to fuse image features with metadata:
```
Image ──► EfficientNet-B4 ──► fc(2048→512) ──┐
                                               ├─► cat ─► classifier ─► 7 classes
Metadata ──► MLP(n→64) ─────────────────────┘
```

In [1]:
import sys
sys.path.insert(0, '..')
import os, torch, numpy as np, pandas as pd
import torch.optim as optim

from src.dataset  import build_dataloaders, compute_class_weights, CLASS_NAMES, get_metadata_dim, engineer_metadata
from src.models   import build_model, get_param_groups
from src.losses   import build_loss
from src.train    import train, build_scheduler, evaluate
from src.evaluate import evaluate_model, ensemble_predict, plot_confusion_matrix, plot_roc_curves

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT = '../dataset'
print(f'Device: {DEVICE}')

Device: cuda


In [2]:
train_loader, val_loader, test_loader, test_df = build_dataloaders(
    data_root=DATA_ROOT, image_size=224, batch_size=32,
    use_oversampling=True, use_mixup_cutmix=True,
)

meta_df = pd.read_csv(f'{DATA_ROOT}/HAM10000_metadata.csv')
labels_all = [CLASS_NAMES.index(x) for x in meta_df['dx']]
class_weights = compute_class_weights(labels_all).to(DEVICE)

[Dataset] Train: 7054 | Val: 1464 | Test: 1497
[Dataset] Class dist (train): {'nv': np.int64(4730), 'mel': np.int64(777), 'bkl': np.int64(775), 'bcc': np.int64(365), 'akiec': np.int64(233), 'vasc': np.int64(98), 'df': np.int64(76)}


## 1. Train ResNet-50 + CBAM

In [3]:
resnet = build_model('resnet50_cbam', num_classes=7, pretrained=True).to(DEVICE)
criterion_res = build_loss('focal', class_weights=class_weights, gamma=2.0)
param_groups_res = get_param_groups(resnet, base_lr=1e-4, backbone_lr_multiplier=0.1)
optimizer_res = optim.AdamW(param_groups_res, weight_decay=1e-4)
scheduler_res = build_scheduler(optimizer_res, warmup_epochs=3, total_epochs=40)

history_res = train(
    model=resnet, train_loader=train_loader, val_loader=val_loader,
    criterion=criterion_res, optimizer=optimizer_res, scheduler=scheduler_res,
    device=DEVICE, num_epochs=40, freeze_epochs=3, patience=10,
    checkpoint_dir='../checkpoints', model_name='resnet50_cbam'
)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\ghosh/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:08<00:00, 11.5MB/s]
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:201: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


[Train] Freezing backbone for 3 epochs...


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 001/40 | LR: 3.40e-05 | Train Loss: 1.5071  Acc: 0.1388  BalAcc: 0.1383 | Val Loss: 0.3105  Acc: 0.2527  BalAcc: 0.1789 | 46.6s
  ✓ New best | BalAcc: 0.1789 → saved to ../checkpoints\resnet50_cbam_best.pth


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


...

Epoch 039/40 | LR: 9.46e-05 | Train Loss: 0.5187  Acc: 0.6926  BalAcc: 0.6920 | Val Loss: 0.0918  Acc: 0.7725  BalAcc: 0.7381 | 64.0s


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 040/40 | LR: 9.27e-05 | Train Loss: 0.4970  Acc: 0.7070  BalAcc: 0.7060 | Val Loss: 0.0874  Acc: 0.7821  BalAcc: 0.7318 | 64.5s


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:269: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)


[Train] Loaded best checkpoint (epoch 37, BalAcc: 0.7518)


## 2. Train DenseNet-169

In [4]:
densenet = build_model('densenet169', num_classes=7, pretrained=True).to(DEVICE)
criterion_den = build_loss('label_smoothing', class_weights=class_weights)
param_groups_den = get_param_groups(densenet, base_lr=1e-4, backbone_lr_multiplier=0.1)
optimizer_den = optim.AdamW(param_groups_den, weight_decay=1e-4)
scheduler_den = build_scheduler(optimizer_den, warmup_epochs=3, total_epochs=40)

history_den = train(
    model=densenet, train_loader=train_loader, val_loader=val_loader,
    criterion=criterion_den, optimizer=optimizer_den, scheduler=scheduler_den,
    device=DEVICE, num_epochs=40, freeze_epochs=3, patience=10,
    checkpoint_dir='../checkpoints', model_name='densenet169'
)

Downloading: "https://download.pytorch.org/models/densenet169-b2777c0a.pth" to C:\Users\ghosh/.cache\torch\hub\checkpoints\densenet169-b2777c0a.pth
100%|██████████| 54.7M/54.7M [00:06<00:00, 9.48MB/s]
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:201: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


[Train] Freezing backbone for 3 epochs...


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 001/40 | LR: 3.40e-05 | Train Loss: 2.1375  Acc: 0.1479  BalAcc: 0.1459 | Val Loss: 0.6092  Acc: 0.1018  BalAcc: 0.1292 | 65.1s
  ✓ New best | BalAcc: 0.1292 → saved to ../checkpoints\densenet169_best.pth


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


...

Epoch 039/40 | LR: 9.46e-05 | Train Loss: 0.8534  Acc: 0.5763  BalAcc: 0.5737 | Val Loss: 0.4100  Acc: 0.3149  BalAcc: 0.6534 | 51.5s


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 040/40 | LR: 9.27e-05 | Train Loss: 0.8372  Acc: 0.5709  BalAcc: 0.5688 | Val Loss: 0.4113  Acc: 0.3292  BalAcc: 0.6657 | 51.5s


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:269: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)


[Train] Loaded best checkpoint (epoch 33, BalAcc: 0.6700)


## 3. Soft-Voting Ensemble

In [5]:
# Load all best checkpoints
effnet = build_model('efficientnet_b4', num_classes=7).to(DEVICE)
ckpt_e = torch.load('../checkpoints/efficientnet_b4_best.pth', map_location=DEVICE)
effnet.load_state_dict(ckpt_e['model_state_dict'])

ckpt_r = torch.load('../checkpoints/resnet50_cbam_best.pth', map_location=DEVICE)
resnet.load_state_dict(ckpt_r['model_state_dict'])

ckpt_d = torch.load('../checkpoints/densenet169_best.pth', map_location=DEVICE)
densenet.load_state_dict(ckpt_d['model_state_dict'])

# Ensemble predict
ens_probs, ens_preds, ens_targets = ensemble_predict(
    models=[effnet, resnet, densenet],
    loader=test_loader, device=DEVICE, method='soft_voting'
)

print('\n=== ENSEMBLE RESULTS ===')
ens_metrics = evaluate_model(
    ens_probs, ens_preds, ens_targets,
    bootstrap_ci=True, save_dir='../results'
)
plot_confusion_matrix(ens_targets, ens_preds, '../results/cm_ensemble.png')
plot_roc_curves(ens_probs, ens_targets, '../results/roc_ensemble.png')

C:\Users\ghosh\AppData\Local\Temp\ipykernel_27324\3142070586.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt_e = torch.load('../checkpoints/efficientnet_b4_best.pth


=== ENSEMBLE RESULTS ===
Balanced Accuracy: 0.7938 (95% CI: 0.7573–0.8247)

  Accuracy:          0.6780
  Balanced Accuracy: 0.7938
  Cohen's Kappa:     0.7062
  AUC (macro OvR):   0.9576
  AUC (weighted):    0.9436
              precision    recall  f1-score   support

       akiec       0.47      0.90      0.62        51
         bcc       0.56      0.82      0.67        77
         bkl       0.65      0.58      0.61       157
          df       0.53      0.86      0.66        22
         mel       0.33      0.80      0.46       168
          nv       0.98      0.64      0.78      1000
        vasc       0.47      0.95      0.63        22

    accuracy                           0.68      1497
   macro avg       0.57      0.79      0.63      1497
weighted avg       0.82      0.68      0.71      1497

[Eval] Saved confusion matrix → ../results/cm_ensemble.png
[Eval] Saved ROC curves → ../results/roc_ensemble.png


## 4. Multi-Modal Fusion (Image + Metadata)

In [6]:
# Load data WITH metadata
train_mm, val_mm, test_mm, _ = build_dataloaders(
    data_root=DATA_ROOT, image_size=224, batch_size=32,
    use_oversampling=True, use_mixup_cutmix=False,
    return_metadata=True,
)

# Compute metadata dimension
meta_df_eng = engineer_metadata(pd.read_csv(f'{DATA_ROOT}/HAM10000_metadata.csv'))
meta_dim = get_metadata_dim(meta_df_eng)
print(f'Metadata feature dimension: {meta_dim}')

# Build multi-modal model
mm_model = build_model('multimodal', num_classes=7, metadata_dim=meta_dim,
                        pretrained=True).to(DEVICE)

criterion_mm = build_loss('focal', class_weights=class_weights, gamma=2.0)
optimizer_mm = optim.AdamW(mm_model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_mm = build_scheduler(optimizer_mm, warmup_epochs=3, total_epochs=35)

history_mm = train(
    model=mm_model, train_loader=train_mm, val_loader=val_mm,
    criterion=criterion_mm, optimizer=optimizer_mm, scheduler=scheduler_mm,
    device=DEVICE, num_epochs=35, freeze_epochs=3, patience=10,
    checkpoint_dir='../checkpoints', model_name='multimodal_fusion',
    is_multimodal=True,
)

[Dataset] Train: 7054 | Val: 1464 | Test: 1497
[Dataset] Class dist (train): {'nv': np.int64(4730), 'mel': np.int64(777), 'bkl': np.int64(775), 'bcc': np.int64(365), 'akiec': np.int64(233), 'vasc': np.int64(98), 'df': np.int64(76)}
Metadata feature dimension: 19


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:201: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


[Train] Freezing backbone for 3 epochs...


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 001/35 | LR: 3.40e-05 | Train Loss: 1.4121  Acc: 0.1489  BalAcc: 0.1476 | Val Loss: 0.3127  Acc: 0.0157  BalAcc: 0.1398 | 43.1s
  ✓ New best | BalAcc: 0.1398 → saved to ../checkpoints\multimodal_fusion_best.pth


...

Epoch 034/35 | LR: 9.98e-05 | Train Loss: 0.0757  Acc: 0.7675  BalAcc: 0.7658 | Val Loss: 0.1093  Acc: 0.4788  BalAcc: 0.7241 | 51.5s


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 035/35 | LR: 9.94e-05 | Train Loss: 0.0729  Acc: 0.7726  BalAcc: 0.7718 | Val Loss: 0.1153  Acc: 0.5273  BalAcc: 0.7451 | 51.4s


d:\Code\student_works\Skin_Lesion\notebooks\..\src\train.py:269: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)


[Train] Loaded best checkpoint (epoch 32, BalAcc: 0.7474)


In [7]:
# Evaluate multi-modal model
ckpt_mm = torch.load('../checkpoints/multimodal_fusion_best.pth', map_location=DEVICE)
mm_model.load_state_dict(ckpt_mm['model_state_dict'])

test_criterion_mm = build_loss('label_smoothing', class_weights=class_weights)
mm_test_metrics = evaluate(mm_model, test_mm, test_criterion_mm, DEVICE, is_multimodal=True)

print('\n=== MULTIMODAL FUSION RESULTS ===')
mm_metrics = evaluate_model(
    mm_test_metrics['all_probs'], mm_test_metrics['all_preds'],
    mm_test_metrics['all_targets'], bootstrap_ci=True, save_dir='../results'
)
plot_confusion_matrix(mm_test_metrics['all_targets'], mm_test_metrics['all_preds'],
                      '../results/cm_multimodal.png')

C:\Users\ghosh\AppData\Local\Temp\ipykernel_27324\611056345.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt_mm = torch.load('../checkpoints/multimodal_fusion_best.p


=== MULTIMODAL FUSION RESULTS ===
Balanced Accuracy: 0.7198 (95% CI: 0.6773–0.7565)

  Accuracy:          0.4703
  Balanced Accuracy: 0.7198
  Cohen's Kappa:     0.5587
  AUC (macro OvR):   0.9201
  AUC (weighted):    0.8994
              precision    recall  f1-score   support

       akiec       0.38      0.86      0.53        51
         bcc       0.50      0.82      0.62        77
         bkl       0.39      0.55      0.46       157
          df       0.26      0.82      0.40        22
         mel       0.22      0.73      0.33       168
          nv       0.99      0.35      0.52      1000
        vasc       0.43      0.91      0.59        22

    accuracy                           0.47      1497
   macro avg       0.45      0.72      0.49      1497
weighted avg       0.78      0.47      0.50      1497

[Eval] Saved confusion matrix → ../results/cm_multimodal.png


## 5. Model Comparison Table

In [8]:
# Evaluate individual models for fair comparison
import json

comparison = pd.DataFrame(columns=['Model', 'Balanced Acc', 'AUC (macro)', "Cohen's Kappa", 'Accuracy'])

print('Comparison table will be populated after all models are trained.')
print('Expected performance range (HAM10000 literature):')
print(pd.DataFrame([
    {'Model': 'EfficientNet-B4',   'Balanced Acc (lit)': '~0.83', 'AUC (lit)': '~0.97'},
    {'Model': 'ResNet50+CBAM',     'Balanced Acc (lit)': '~0.80', 'AUC (lit)': '~0.96'},
    {'Model': 'DenseNet-169',      'Balanced Acc (lit)': '~0.81', 'AUC (lit)': '~0.96'},
    {'Model': 'ViT-B/16',          'Balanced Acc (lit)': '~0.82', 'AUC (lit)': '~0.97'},
    {'Model': 'Ensemble (3 model)','Balanced Acc (lit)': '~0.86', 'AUC (lit)': '~0.98'},
    {'Model': 'Multimodal Fusion', 'Balanced Acc (lit)': '~0.84', 'AUC (lit)': '~0.97'},
]).to_string(index=False))

Comparison table will be populated after all models are trained.
Expected performance range (HAM10000 literature):
             Model Balanced Acc (lit) AUC (lit)
   EfficientNet-B4              ~0.83     ~0.97
     ResNet50+CBAM              ~0.80     ~0.96
      DenseNet-169              ~0.81     ~0.96
          ViT-B/16              ~0.82     ~0.97
Ensemble (3 model)              ~0.86     ~0.98
 Multimodal Fusion              ~0.84     ~0.97
